<style>
.custom-spacing {
    line-height: 0.5;
}
</style>
### <center>**KECERDASAN ARTIFISIAL UNTUK GEOFISIKA**</center>
#### <center>**TUGAS INDIVIDU**</center>


<div style="text-align:center;">
  <img src="https://www.unhas.ac.id/wp-content/uploads/2022/09/Logo-Resmi-Unhas-1.png" alt="Deskripsi Gambar" style="width:280px;">
</div>

#### <center>**DISUSUN OLEH :**</center>

#### <center>**PUTRI MEYLANI          (H061241088)**<center>

### <center>**PROGRAM STUDI GEOFISIKA**</center>
### <center>**FAKULTAS MATEMATIKA DAN ILMU PENGETAHUAN ALAM**</center>
#### <center>**UNIVERSITAS HASANUDDIN**</center>
#### <center>**MAKASSAR**</center>
#### <center>**2026**</center>

<hr>

# Form Prompt AI untuk Geofisika

Struktur mengikuti Bab 2 *AI Cookbook untuk Earth Sciences* (prompt sebagai *work specification*).

| Kode | Komponen | Fungsi |
|------|----------|--------|
| **R** | Role | Sudut pandang kerja, bukan credential |
| **T** | Task | Verb yang bisa diperiksa + objek + domain + kondisi penerimaan |
| **C** | Context | Tujuan, audiens, workflow, keputusan yang didukung |
| **I** | Input | Path, format, variabel, unit, CRS, provenance |
| **K** | Constraints | File yang tidak boleh diubah, method, package, jaringan, approval |
| **O** | Output | Kontrak output: path, kolom/unit, format (rencana teks, belum hasil) |
| **Q** | Quality criteria | Kriteria yang dapat dinilai eksplisit |
| **V** | Optional verification | Check + expected + tindakan bila gagal |

> Bagian **O** adalah *expected output*, bukan *actual result*.

---

## Template Umum

```markdown
## [R] Role
Bertindaklah sebagai <sudut pandang kerja>. Fokus pada <apa yang diperiksa>.

## [T] Task
<Verb> <objek> <domain/kondisi>, lalu <verb kedua>. <Kondisi penerimaan>.

## [C] Context
<Tujuan>, <audiens>, <tahap workflow>. Hasil dipakai untuk <keputusan>, bukan untuk <hal yang tidak dilakukan>.

## [I] Input
- `<path>`: format, variabel/kolom, unit, CRS/datum, sumber
- Jangan memakai: <input terlarang>

## [K] Constraints
- Jangan mengubah <file/folder>.
- Method: <yang boleh>; <yang dilarang>.
- Software: <yang tersedia>; tanpa jaringan.
- Bila <kondisi tak jelas>: berhenti dan tanyakan.

## [O] Output (rencana teks)
- `<path output>`: <isi, kolom, unit>

## [Q] Quality criteria
- <kriteria terukur>

## [V] Optional verification
- Check: <command>. Expected: <kondisi>. Jika gagal: <tindakan>.
```


## Prompt 1 — QC dan Interpretasi Awal VES Geolistrik Schlumberger

### [R] Role
Bertindaklah sebagai reviewer QC data geolistrik dengan latar belakang geofisika hidrogeologi, terbiasa memeriksa data lapangan VES sebelum masuk tahap inversi. Fokus kerja: (1) konsistensi geometri elektroda AB/2 dan MN/2 terhadap asumsi konfigurasi Schlumberger, (2) kewajaran nilai arus, tegangan, dan resistivitas semu terhadap rentang fisik yang masuk akal, (3) traceability setiap angka ρa ke rumus faktor geometri yang dipakai, dan (4) pemisahan tegas antara "data yang diverifikasi", "data yang diasumsikan", dan "data yang tidak dapat diperiksa". Kamu tidak berperan sebagai interpreter geologi atau juru bor pada tahap ini — keputusan soal lapisan akuifer dan lokasi bor bukan bagian dari peranmu.

### [T] Task
Periksa data sounding `data/raw/ves_line01.csv` untuk konsistensi AB/2, MN/2, arus, dan tegangan, hitung ulang resistivitas semu (ρa) dari `ΔV/I × faktor geometri`, lalu tandai titik yang menyimpang dari kurva sounding yang monoton wajar. Jangan menjalankan inversi otomatis atau menetapkan model lapisan akhir.

### [C] Context
Sounding dilakukan di `{{lokasi}}` untuk eksplorasi air tanah dangkal. Hasil QC dipakai tim lapangan untuk memutuskan apakah titik perlu diulang sebelum interpretasi litologi dan rekomendasi bor. Hasil tahap ini bukan penentuan lapisan akuifer final.

### [I] Input
- `data/raw/ves_line01.csv`: kolom `ab_2_m`, `mn_2_m`, `i_ma`, `dv_mv`, `rho_a_ohm_m` (bila sudah dihitung sumber). Konfigurasi: Schlumberger. Sumber alat: `{{merek/model resistivitymeter}}`.
- `data/raw/ves_coordinates.csv`: `point_id`, `lat`, `lon`, `elev_m`, CRS `{{EPSG:...}}`.
- Jangan memakai: nilai resistivitas dari laporan pihak lain yang tidak tercantum di file sumber.

### [K] Constraints
- Jangan mengubah `data/raw/`; jangan menghapus baris pengukuran.
- Jangan menjalankan inversi 1D/2D, memilih jumlah lapisan, atau menetapkan model geologi.
- Gunakan hanya Python standard library, NumPy, dan pandas; tanpa akses jaringan.
- Ambang kewajaran ρa (mis. lompatan tajam antar titik berdekatan) harus dinyatakan sebagai `{{ambang %}}`, bukan ditetapkan sendiri tanpa disebut.
- Jika AB/2 dan MN/2 tidak konsisten dengan asumsi Schlumberger (MN ≪ AB), tandai "tidak terverifikasi" dan berhenti pada titik itu.

### [O] Output (rencana teks)
- `results/ves/ves_line01_qc.md`: jumlah titik sounding, rentang AB/2, jumlah titik dengan flag, ringkasan bentuk kurva per segmen.
- `results/ves/ves_line01_flags.csv`: `point_id`, `flag_type`, `value`, `threshold`, `note`.
- `results/ves/rho_a_recomputed.csv`: `point_id`, `rho_a_input`, `rho_a_recomputed`, `selisih_persen`.
- `results/ves/summary.json`: `n_points`, `n_flagged`, `n_non_monotonic`.

### [Q] Quality criteria
- Jumlah baris `flags.csv` + baris tak bermasalah = jumlah titik input.
- Setiap flag memuat nilai, ambang, dan alasan.
- `rho_a_recomputed` dan `rho_a_input` dibandingkan secara eksplisit, bukan diasumsikan sama.
- Raw data tidak berubah.

### [V] Optional verification
- Check: hitung ulang faktor geometri Schlumberger `π[(AB/2)² − (MN/2)²]/MN` untuk 5 titik acak. Expected: selisih dengan `rho_a_recomputed` di bawah `{{toleransi}}`%. Jika gagal: tandai rumus faktor geometri tidak konsisten.
- Check: SHA-256 `ves_line01.csv` sebelum dan sesudah prosedur. Expected: identik. Jika gagal: hentikan dan laporkan.

## Prompt 2 — QC Inversi ERT 2D untuk Kandidat Bidang Gelincir

### [R] Role
Bertindaklah sebagai reviewer QC hasil inversi resistivitas 2D dengan latar belakang geofisika geoteknik, terbiasa menilai keluaran software inversi (bukan menjalankannya) untuk keperluan mitigasi bencana. Fokus kerja: (1) menilai kualitas fit inversi melalui RMS error dan jumlah iterasi, bukan asumsi bahwa RMS rendah berarti model benar secara geologi, (2) memeriksa konsistensi satuan resistivitas, kedalaman, dan jarak sepanjang lintasan, (3) memisahkan secara eksplisit kalimat fakta terukur (nilai resistivitas, koordinat, RMS) dari kalimat dugaan geologi (litologi, bidang gelincir), dan (4) menahan diri dari membuat kesimpulan rekayasa seperti faktor keamanan lereng, karena itu di luar kewenangan geofisikawan dan memerlukan analisis geoteknik terpisah.

### [T] Task
Periksa file hasil inversi `data/raw/ert_line02_inversion.csv` untuk RMS error, sebaran resistivitas per segmen jarak dan kedalaman, lalu tandai segmen dengan resistivitas rendah yang berdekatan dengan permukaan sebagai kandidat zona jenuh air. Jangan menyimpulkan bidang gelincir atau faktor keamanan lereng.

### [C] Context
Lereng di `{{lokasi}}` mengalami retakan tanah setelah hujan lebat. Hasil dipakai tim mitigasi bencana daerah sebagai informasi awal untuk memutuskan survei lanjutan (bor, inklinometer), bukan sebagai analisis stabilitas lereng final.

### [I] Input
- `data/raw/ert_line02_inversion.csv`: `x_m`, `z_m` (kedalaman, m), `resistivity_ohm_m`, `sensitivity` (bila ada).
- `data/raw/ert_line02_meta.json`: konfigurasi (`Wenner-Schlumberger`), spasi elektroda `{{m}}`, panjang lintasan `{{m}}`, RMS error inversi `{{%}}`, jumlah iterasi.
- Ambang resistivitas rendah yang dianggap dugaan zona jenuh: `{{ohm·m}}`.

### [K] Constraints
- Jangan mengubah `data/raw/`; jangan menjalankan ulang inversi atau mengubah parameter regularisasi.
- Bedakan fakta (nilai resistivitas dan RMS) dari dugaan (litologi, bidang gelincir); tulis dugaan sebagai hipotesis eksplisit.
- Jika RMS error di atas `{{ambang %}}`, beri peringatan keandalan model pada laporan, jangan menyembunyikannya.
- Software: Python, NumPy, pandas, matplotlib yang tersedia; tanpa jaringan.
- Jangan menetapkan faktor keamanan lereng dalam bentuk apa pun.

### [O] Output (rencana teks)
- `results/ert/ert_line02_qc.md`: penilaian kualitas inversi, jumlah segmen, ringkasan RMS.
- `results/ert/zona_kandidat.csv`: `x_m`, `z_top_m`, `z_bottom_m`, `resistivity_ohm_m`, `kategori` (rendah/sedang/tinggi), `catatan_dugaan`.
- `results/ert/run_config.json`: ambang resistivitas, konfigurasi elektroda, RMS error, versi package.
- `results/ert/rekomendasi_lanjutan.md`: daftar segmen prioritas untuk bor/inklinometer beserta alasan.

### [Q] Quality criteria
- Setiap segmen kandidat memuat koordinat x, rentang kedalaman, dan nilai resistivitas yang mendasarinya.
- Kalimat dugaan geologi ditulis terpisah dari kalimat fakta terukur.
- RMS error dan keterbatasannya dinyatakan eksplisit di laporan.
- Tidak ada klaim faktor keamanan atau probabilitas longsor.

### [V] Optional verification
- Check: bandingkan jumlah segmen berkategori "rendah" dengan histogram resistivitas mentah. Expected: konsisten dengan ambang yang dinyatakan. Jika gagal: tandai kategori tidak konsisten.
- Check: SHA-256 file input sebelum dan sesudah. Expected: identik. Jika gagal: hentikan dan laporkan.

## Prompt 3 — Pengolahan Data Seismik Refraksi untuk Kedalaman Bedrock

### [R] Role
Bertindaklah sebagai insinyur geofisika seismik dangkal yang biasa menangani survei refraksi untuk keperluan rekayasa sipil, bukan eksplorasi migas atau mineral dalam. Fokus kerja: (1) membaca travel-time curve secara hati-hati untuk menentukan titik patah (crossover) tanpa memaksakan jumlah lapisan yang tidak didukung data, (2) menghitung kecepatan tiap segmen dengan rumus yang ditampilkan secara eksplisit beserta substitusi angkanya, bukan hasil akhir saja, (3) menjaga konsistensi satuan antara meter, milidetik, dan meter/detik di seluruh perhitungan, dan (4) melaporkan kedalaman bedrock sebagai estimasi dengan rentang ketidakpastian, bukan angka pasti, karena hasil ini akan dipakai tim sipil sebagai salah satu input desain, bukan satu-satunya dasar keputusan.

### [T] Task
Olah data `data/raw/refraction_ttcurve.csv` (jarak geophone vs waktu tiba, shot maju dan mundur) untuk menghitung kecepatan tiap lapisan (V1, V2, dan V3 bila ada) serta kedalaman bedrock dengan metode intercept time atau reciprocal. Jangan mengekstrapolasi ke lapisan yang datanya tidak mencukupi.

### [C] Context
Survei mendukung perencanaan fondasi bangunan di `{{lokasi}}`. Tim teknik sipil membutuhkan kedalaman lapisan keras dan indikasi rippability sebagai salah satu masukan desain, bukan sebagai penentu tunggal.

### [I] Input
- `data/raw/refraction_ttcurve.csv`: `shot_id`, `shot_x_m`, `receiver_x_m`, `t_fb_ms`, `arah` (forward/reverse).
- Spasi geophone: `{{m}}`; jumlah geophone: `{{n}}`.
- Ambang reciprocal time yang dapat diterima: `{{ms}}`.

### [K] Constraints
- Jangan mengubah file input; jangan melakukan picking ulang.
- Tampilkan rumus dan substitusi angka pada setiap perhitungan kecepatan dan kedalaman.
- Jika crossover distance tidak jelas atau data tidak cukup untuk lapisan ke-3, nyatakan dengan jelas dan hentikan pada bagian itu.
- Software: Python, NumPy, pandas; tanpa jaringan.
- Satuan konsisten (m, ms, m/s) pada seluruh output.

### [O] Output (rencana teks)
- `results/refraction/velocity_model.csv`: `segment`, `v_m_s`, `intercept_time_ms`, `crossover_distance_m`.
- `results/refraction/bedrock_depth.csv`: `shot_id`, `depth_m`, `metode`, `catatan_ketidakpastian`.
- `results/refraction/reciprocal_check.csv`: `shot_pair`, `t_forward_ms`, `t_reverse_ms`, `diff_ms`, `status`.
- `results/refraction/run_config.json`: metode, rumus, parameter geometri.

### [Q] Quality criteria
- Semua angka kecepatan dan kedalaman dapat ditelusuri ke rumus dan data input.
- Kedalaman bedrock disertai kisaran ketidakpastian, bukan angka tunggal tanpa konteks.
- Reciprocal time dilaporkan untuk setiap pasangan shot yang tersedia.
- Klasifikasi rippability (bila diminta) dinyatakan sebagai indikasi, bukan kepastian.

### [V] Optional verification
- Check: hitung ulang kedalaman bedrock untuk 3 shot point secara independen. Expected: selisih di bawah `{{toleransi}}`%. Jika gagal: laporkan rumus atau data yang berbeda.
- Check: bandingkan `t_forward_ms` dan `t_reverse_ms` pada seluruh pasangan reciprocal. Expected: `diff_ms` di bawah ambang. Jika gagal: tandai lintasan tersebut sebagai tidak reciprocal-consistent.

## Prompt 4 — Koreksi Gravitasi dan Anomali Bouguer untuk Struktur Cekungan

### [R] Role
Bertindaklah sebagai reviewer implementasi numerik untuk data gravitasi eksplorasi, dengan tanggung jawab memastikan pipeline koreksi benar sebelum anomali dibaca oleh tim interpretasi. Fokus kerja: (1) memverifikasi urutan koreksi sesuai konvensi (drift → pasang surut bila ada → lintang normal → free-air → Bouguer sederhana → terrain), karena urutan yang salah mengubah nilai CBA secara signifikan, (2) memastikan setiap konstanta (densitas reduksi, rumus lintang normal, tinggi upward continuation) tercatat sebagai parameter yang bisa diaudit, bukan angka tersembunyi dalam kode, (3) menjaga agar setiap kolom hasil punya satuan eksplisit dalam mGal, dan (4) menahan diri dari menafsirkan sumber anomali sebagai struktur geologi tertentu, apalagi hidrokarbon atau mineral — itu wewenang tahap interpretasi terpisah dengan data pendukung tambahan.

### [T] Task
Hitung anomali Bouguer lengkap (CBA) untuk setiap stasiun dari `data/raw/gravity_cekungan.csv` dengan urutan koreksi drift, lintang normal, free-air, Bouguer sederhana, dan terrain, lalu pisahkan anomali regional (upward continuation) dari residual. Jangan menginterpretasikan sumber anomali sebagai hidrokarbon atau mineral tertentu.

### [C] Context
Penelitian awal eksplorasi cekungan di `{{lokasi}}` dengan `{{jumlah}}` titik pengukuran. Hasil dipakai untuk memilih area prioritas survei seismik lanjutan, bukan sebagai keputusan eksplorasi final.

### [I] Input
- `data/raw/gravity_cekungan.csv`: `station_id`, `lat`, `lon` (WGS84), `elev_m`, `g_obs_mgal`, `time_utc`, `terrain_corr_mgal` (bila ada).
- `data/raw/base_station.csv`: nilai absolut base `{{mGal}}`, drift `{{mGal/jam}}`.
- Densitas reduksi: `{{2.67 g/cm³}}`. Ketinggian upward continuation: `{{h}}` m.

### [K] Constraints
- Urutan koreksi harus dinyatakan eksplisit dan tidak diubah tanpa alasan.
- Jangan mengubah densitas reduksi secara diam-diam; tulis sebagai parameter di `run_config.json`.
- Jangan memakai stasiun tanpa data elevasi; keluarkan ke tabel flag.
- Software: Python, NumPy, pandas, matplotlib yang tersedia; jangan memasang package baru.
- Jangan mengklaim keberadaan hidrokarbon atau mineral tertentu.

### [O] Output (rencana teks)
- `results/gravity/bouguer.csv`: `station_id`, `g_corr_drift`, `g_normal`, `free_air`, `bouguer_simple`, `terrain`, `bouguer_complete`, `regional`, `residual` (semua mGal).
- `results/gravity/run_config.json`: densitas, rumus lintang normal, tinggi upward continuation, versi package.
- `results/gravity/summary.md`: jumlah stasiun, statistik tiap kolom, stasiun yang di-flag.
- `results/gravity/flags.csv`: stasiun yang tidak dapat diproses beserta alasan.

### [Q] Quality criteria
- Semua nilai anomali finite untuk stasiun yang diproses.
- Unit setiap kolom eksplisit (mGal).
- Jumlah stasiun output + flag = input.
- Rumus dan konstanta tercatat di `run_config.json`.

### [V] Optional verification
- Check: hitung ulang 5 stasiun secara independen. Expected: selisih di bawah `{{0.01 mGal}}`. Jika gagal: laporkan rumus yang berbeda.
- Check: korelasi residual dengan elevasi. Expected: tidak ada korelasi kuat. Jika ada: laporkan kemungkinan koreksi terrain kurang memadai, jangan diperbaiki sendiri.

## Prompt 5 — QC dan Pra-pengolahan Data Magnetik untuk Deteksi Struktur Terpendam

### [R] Role
Bertindaklah sebagai reviewer QC data magnetik yang bertugas menyiapkan data bersih sebelum diserahkan ke interpreter manusia, bukan menjadi interpreter itu sendiri. Fokus kerja: (1) memverifikasi bahwa koreksi variasi harian memakai data base station yang benar-benar menaungi seluruh waktu survei, bukan diekstrapolasi diam-diam, (2) memastikan pengurangan medan referensi IGRF memakai model dan epoch yang dinyatakan pengguna, tanpa mengunduh atau mengganti model sendiri, (3) mendeteksi spike dan gap pada grid dengan ambang yang eksplisit dan dapat diaudit, bukan penilaian subjektif, dan (4) secara sadar tidak melangkah ke reduksi ke kutub (RTP) atau estimasi kedalaman sumber, karena RTP di lintang rendah rawan artefak dan keputusan itu perlu pertimbangan manusia dengan konteks geologi tambahan.

### [T] Task
Periksa dan kurangkan variasi harian serta medan referensi IGRF dari data magnetik total `data/raw/mag_terpendam.csv`, lalu buat grid anomali residual dan tandai spike serta gap. Jangan menjalankan reduksi ke kutub atau menentukan jenis mineral/objek terpendam.

### [C] Context
Survei magnetik di lahan `{{lokasi}}` yang diduga mengandung mineral bijih besi atau struktur logam terpendam. Hasil grid menjadi input tahap interpretasi yang dilakukan manusia untuk menentukan titik bor verifikasi; tahap ini hanya menyiapkan data bersih.

### [I] Input
- `data/raw/mag_terpendam.csv`: `point_id`, `lat`, `lon` (WGS84), `time_utc`, `B_nT`, ketinggian sensor `{{m}}`.
- `data/raw/base_mag.csv`: `time_utc`, `B_base_nT`.
- IGRF: `{{model dan epoch, mis. IGRF-13 2020}}` dari `{{file/nilai yang diberikan}}`.
- Grid: sel `{{m}}`, metode `{{minimum curvature}}`.

### [K] Constraints
- Jangan mengubah `data/raw/`.
- Jangan menjalankan RTP/RTE atau estimasi kedalaman sumber; catat bahwa RTP di lintang rendah butuh keputusan manusia.
- Jangan menghapus spike; hanya beri flag dengan ambang `{{nT}}` yang dinyatakan.
- Jangan mengunduh model IGRF; gunakan nilai yang diberikan.
- Jika interval waktu base station tidak mencakup survei, hentikan dan laporkan.

### [O] Output (rencana teks)
- `results/mag/anomaly.csv`: `point_id`, `B_corr_nT`, `IGRF_nT`, `anomaly_nT`, `flag`.
- `results/mag/grid_residual.csv` (atau `.tif`): grid anomali dengan CRS dan ukuran sel tercatat.
- `results/mag/qc_report.md`: jumlah titik, spike, gap, cakupan waktu base.
- `results/mag/run_config.json`: ambang spike, model IGRF, metode grid.

### [Q] Quality criteria
- Jumlah titik input sama dengan output (termasuk yang di-flag).
- Nilai anomali finite; satuan nT.
- Semua parameter yang memengaruhi hasil tercatat.
- Tidak ada interpretasi jenis mineral atau objek pada laporan QC ini.

### [V] Optional verification
- Check: bandingkan statistik anomali sebelum dan sesudah koreksi harian. Expected: rentang lebih kecil setelah koreksi. Jika tidak: laporkan dan periksa sinkronisasi waktu.
- Check: cek titik di tepi grid. Expected: tidak ada ekstrapolasi jauh dari data. Jika ada: beri flag pada sel tersebut.